# RAG Application Example - Code Analysis

This notebook demonstrates a real-world RAG application for analyzing code files.
The example shows how to query code repositories and get intelligent responses about code patterns, best practices, and potential issues.

In [9]:
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

True

In [10]:
from llama_index.llms.nvidia import NVIDIA
from llama_index.core import Settings

# Initialize with an NVIDIA model (e.g., Llama 3)
llm = NVIDIA(model="openai/gpt-oss-120b")
Settings.llm = llm

print(f"Using LLM: {llm.model}")

Using LLM: openai/gpt-oss-120b


In [11]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core import Settings

embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")
Settings.embed_model = embedder

print(f"Using Embedding model: {embedder.model}")

Using Embedding model: nvidia/nv-embedqa-e5-v5


In [12]:
from pathlib import Path

from dotenv import load_dotenv
from llama_index.llms.nvidia import NVIDIA
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.text_splitter import TokenTextSplitter

# Load environment variables
load_dotenv(dotenv_path='.env')

# Initialize LLM with NVIDIA model
llm = NVIDIA(model="openai/gpt-oss-120b")

# Initialize embedding model
embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")

# Configure settings
Settings.llm = llm
Settings.embed_model = embedder

# Load documents from the RAG data directory
cwd = Path.cwd()
data_dir = cwd / "data"
if not data_dir.exists():
    data_dir = cwd / "rag_example" / "data"

if not data_dir.exists():
    raise FileNotFoundError(
        f"Directory {data_dir} does not exist. Create it or update the path."
    )

documents = SimpleDirectoryReader(str(data_dir)).load_data()
print(f"Loaded {len(documents)} documents")

# Chunk documents to ensure they fit within embedding model limits
# NVIDIA embedqa-e5-v5 has a 512 token limit
text_splitter = TokenTextSplitter(
    chunk_size=200,          # Smaller chunks improve retrieval precision
    chunk_overlap=40         # Keep overlap so context is preserved
)

nodes = text_splitter.get_nodes_from_documents(documents)
print(f"Split into {len(nodes)} document chunks (max {text_splitter.chunk_size} tokens each)")

# Create vector index from chunked document nodes
index = VectorStoreIndex(nodes, embed_model=embedder)
query_engine = index.as_query_engine(
    similarity_top_k=7,
    response_mode="compact",
)

Loaded 6 documents
Split into 8 document chunks (max 200 tokens each)


## Example Queries

In [13]:
# Example 1: General knowledge from documents
query = "What are the main features of this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What are the main features of this project?

Response:
The project provides a complete Retrieval‑Augmented Generation (RAG) pipeline built around local document processing and NVIDIA‑powered AI services. Its key capabilities include:

- **Local document ingestion** using a simple directory reader that loads files from the `data` folder.  
- **Chunking of text** with a token‑based splitter (default `chunk_size=150`, `chunk_overlap=30`) to keep each piece under the NVIDIA token limit.  
- **Embedding generation** via the NVIDIA `nv‑embedqa‑e5‑v5` model, producing vector representations for every chunk.  
- **Vector store indexing** with `VectorStoreIndex`, which stores the embeddings and enables fast similarity search.  
- **Query engine** configured with `similarity_top_k=7` and a compact response mode, allowing users to ask project‑specific questions and receive concise, context‑aware answers.  

Supporting infrastructure includes environment management with `.venv` and `uv sync

In [14]:
# Example 2: Code implementation questions
query = "How do I implement a Retrieval Augmented Generation system with Python?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I implement a Retrieval Augmented Generation system with Python?

Response:
To build a Retrieval‑Augmented Generation (RAG) pipeline in Python, follow these steps:

1. **Read the source files**  
   ```python
   from llama_index import SimpleDirectoryReader
   documents = SimpleDirectoryReader("data").load_data()
   ```

2. **Chunk the text** – keep each chunk under the model’s token limit and add overlap so the query engine can reconstruct context:  
   ```python
   from llama_index import TokenTextSplitter
   splitter = TokenTextSplitter(chunk_size=150, chunk_overlap=30)
   nodes = splitter.split_documents(documents)
   ```

3. **Embed the chunks** – use the NVIDIA embedding model (e.g., `nvidia/nv-embedqa-e5-v5`). The embedding class is provided by `llama-index-embeddings-nvidia`.  
   ```python
   from llama_index.embeddings.nvidia import NvidiaEmbedding
   embed_model = NvidiaEmbedding(model_name="nvidia/nv-embedqa-e5-v5")
   ```

4. **Create a vector store index** –

In [15]:
# Example 3: Dependency questions
query = "What dependencies are used for the RAG pipeline?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What dependencies are used for the RAG pipeline?

Response:
The RAG pipeline relies on the following packages:

- `langchain`  
- `langchain-nvidia-ai-endpoints`  
- `llama-index-core`  
- `llama-index-embeddings-nvidia`  
- `llama-index-llms-nvidia`  
- `llama-index-multi-modal-llms-nvidia`  
- `python-dotenv`


In [16]:
# Example 4: Installation and setup questions
query = "How do I install and configure this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I install and configure this project?

Response:
1. Clone the repository and change into the project directory.  
2. Create (or activate) the virtual environment that the project expects, e.g.:

```bash
python -m venv .venv
source .venv/bin/activate   # on macOS/Linux
# .venv\Scripts\activate    # on Windows
```

3. Install the exact dependency set defined in the `pyproject.toml` with **uv**:

```bash
uv sync --active
```

   This pulls in the required packages such as `jupyterlab`, `langchain`, the NVIDIA endpoint libraries, the Llama‑Index components, and `python-dotenv`.

4. Add your NVIDIA API key to the environment file:

```text
# .env
NVIDIA_API_KEY=your_key_here
```

   The `.env` file is read by the notebook to authenticate calls to the NVIDIA embedding and LLM services.

5. Launch the Jupyter notebook that drives the RAG pipeline:

```bash
jupyter lab rag_example.ipynb
```

   The notebook will read the documents, split them, embed them with the `nvidia/nv-embed